# Error Detection

The objective of this phase is to conduct a comprehensive analysis of the discrepancies identified during the Comparison phase. Rather than simply knowing that errors exist, we now investigate WHY they occurred and categorize them by type and impact.

After comparison revealed £17,857.56 in financial variance across 1,980 discrepant records, this phase focuses on understanding the root causes and patterns behind these discrepancies.

This step includes:

**Error Categorization & Classification:**
- Systematic grouping of discrepancies by type (data quality, system sync, business logic)
- Financial impact scoring (High/Medium/Low risk categories)
- Frequency analysis to identify the most common error patterns

**Root Cause Analysis:**
- Investigation of underlying causes for each error category
- Pattern detection across products and customer segments
- Correlation analysis between different types of errors

**Validation Effectiveness Review:**
- Analysis of how well the Validation phase detected different error types
- Identification of gaps in current validation processes

## Purpose: 
Transform discrepancy identification into categorized, understood problems with clear root causes. This analysis provides the foundation for the Resolution phase, where remediation strategies will be developed and prioritized.

# 1. Error Categorization and Classification 

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import ast

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # parse validation flags
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else []) # parsing strings in the validation_flags column into actual lists
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.


**Step 1** Aggregate duplicates

In [33]:
print("STEP 1: Aggregating duplicates for consistent analysis")

# aggregation logic same as in comparison phase 
agg_logic_oms = {
    'UnitPrice': 'max',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': lambda x: list(set([item for sublist in x for item in sublist]))
}

# define agg logic for wms
agg_logic_wms = agg_logic_oms.copy()

# apply aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_oms).reset_index()
df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_wms).reset_index()

print(f"OMS after aggregation: {len(df_oms_agg)} unique business transactions.")
print(f"WMS after aggregation: {len(df_wms_agg)} unique business transactions.")

STEP 1: Aggregating duplicates for consistent analysis
OMS after aggregation: 536478 unique business transactions.
WMS after aggregation: 536478 unique business transactions.


**Step 2** Create matched datasets

In [34]:
df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

#variance calculatons
df_matched['UnitPrice_variance'] = df_matched['UnitPrice_wms'] - df_matched['UnitPrice_oms']
df_matched['Total_Value_oms'] = df_matched['Quantity'] * df_matched['UnitPrice_oms']
df_matched['Total_Value_wms'] = df_matched['Quantity'] * df_matched['UnitPrice_wms']
df_matched['Total_Value_variance'] = df_matched['Total_Value_wms'] - df_matched['Total_Value_oms']

print(f"Matched records for analysis: {len(df_matched):,}")
print(f"Total financial variance: £{df_matched['Total_Value_variance'].sum():,.2f}")

Matched records for analysis: 536,478
Total financial variance: £-17,857.56


**Step 3** Error categorization adn classification

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Load data with proper encoding handling
def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # Parse validation flags
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')

print("=== ERROR DETECTION & ROOT CAUSE ANALYSIS ===\n")

# STEP 1: AGGREGATE DUPLICATES (same as in Comparison phase)
print("STEP 1: AGGREGATING DUPLICATES FOR CONSISTENT ANALYSIS\n")

# Define aggregation logic for OMS
agg_logic_oms = {
    'UnitPrice': 'max',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': lambda x: list(set([item for sublist in x for item in sublist]))
}

# Define aggregation logic for WMS
agg_logic_wms = agg_logic_oms.copy()

# Apply aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_oms).reset_index()
df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_wms).reset_index()

print(f"OMS after aggregation: {len(df_oms_agg)} unique business transactions")
print(f"WMS after aggregation: {len(df_wms_agg)} unique business transactions")

# STEP 2: CREATE MATCHED DATASET
print(f"\nSTEP 2: CREATING MATCHED DATASET FOR ERROR ANALYSIS\n")

df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

# Calculate variances
df_matched['UnitPrice_variance'] = df_matched['UnitPrice_wms'] - df_matched['UnitPrice_oms']
df_matched['Total_Value_oms'] = df_matched['Quantity'] * df_matched['UnitPrice_oms']
df_matched['Total_Value_wms'] = df_matched['Quantity'] * df_matched['UnitPrice_wms']
df_matched['Total_Value_variance'] = df_matched['Total_Value_wms'] - df_matched['Total_Value_oms']

print(f"Matched records for analysis: {len(df_matched):,}")
print(f"Total financial variance: £{df_matched['Total_Value_variance'].sum():,.2f}")

# STEP 3: ERROR CATEGORIZATION & CLASSIFICATION
print(f"\n=== ERROR CATEGORIZATION & CLASSIFICATION ===\n")

# 1. SYSTEMATIC GROUPING BY ERROR TYPE
print("1. SYSTEMATIC GROUPING BY ERROR TYPE\n")

def classify_error_type(row):
    """Classify errors into categories based on characteristics"""
    price_var = row['UnitPrice_variance']
    flags_oms = row['validation_flags_oms'] if isinstance(row['validation_flags_oms'], list) else []
    flags_wms = row['validation_flags_wms'] if isinstance(row['validation_flags_wms'], list) else []
    
    # Data Quality Errors - Missing or corrupted data
    if 'Missing_UnitPrice' in flags_wms or price_var <= -1.0:
        return 'Data Quality Error'
    elif 'Invalid_Date' in flags_wms:
        return 'Data Quality Error'
    
    # System Sync Errors - Duplicate records after aggregation
    elif 'Duplicate_Record' in flags_oms or 'Duplicate_Record' in flags_wms:
        return 'System Sync Error'
    
    # Business Logic Errors - Small price differences (rounding, etc.)
    elif 0 < abs(price_var) <= 0.05:
        return 'Business Logic Error'
    
    # Large unexplained variances
    elif abs(price_var) > 0.05:
        return 'Unknown Error'
    
    # No error
    else:
        return 'No Error'

# Apply classification
df_matched['error_type'] = df_matched.apply(classify_error_type, axis=1)

# Count by error type
error_type_counts = df_matched['error_type'].value_counts()
print("Error Distribution by Type:")
print(error_type_counts)
print(f"\nTotal records analyzed: {len(df_matched):,}")

# 2. FINANCIAL IMPACT SCORING
print(f"\n2. FINANCIAL IMPACT SCORING\n")

def classify_financial_impact(variance):
    """Classify financial impact as High/Medium/Low"""
    abs_variance = abs(variance)
    if abs_variance >= 10.0:
        return 'High Risk'
    elif abs_variance >= 1.0:
        return 'Medium Risk'
    elif abs_variance > 0:
        return 'Low Risk'
    else:
        return 'No Risk'

# Apply financial impact classification
df_matched['financial_impact'] = df_matched['Total_Value_variance'].apply(classify_financial_impact)

# Financial impact analysis
financial_impact_summary = df_matched.groupby('financial_impact').agg({
    'Total_Value_variance': ['count', 'sum', 'mean'],
    'UnitPrice_variance': ['min', 'max']
}).round(2)

print("Financial Impact Classification:")
print(financial_impact_summary)

# 3. ERROR TYPE vs FINANCIAL IMPACT CROSS-ANALYSIS
print(f"\n3. ERROR TYPE vs FINANCIAL IMPACT CROSS-ANALYSIS\n")
cross_analysis = pd.crosstab(df_matched['error_type'], 
                            df_matched['financial_impact'], 
                            margins=True)
print(cross_analysis)

# 4. FREQUENCY ANALYSIS - MOST COMMON ERROR PATTERNS
print(f"\n4. FREQUENCY ANALYSIS - MOST COMMON ERROR PATTERNS\n")

# Filter only records with errors
error_records = df_matched[df_matched['error_type'] != 'No Error']

# A. Most affected products (StockCode)
print("A. TOP 10 MOST AFFECTED PRODUCTS:")
if len(error_records) > 0:
    product_errors = error_records.groupby('StockCode').agg({
        'Total_Value_variance': ['count', 'sum'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    product_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Most_Common_Error_Type']
    product_errors = product_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(product_errors.head(10))
else:
    print("No error records found for product analysis")

# B. Most affected invoices
print(f"\nB. TOP 10 MOST AFFECTED INVOICES:")
if len(error_records) > 0:
    invoice_errors = error_records.groupby('InvoiceNo').agg({
        'Total_Value_variance': ['count', 'sum'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    invoice_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Most_Common_Error_Type']
    invoice_errors = invoice_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(invoice_errors.head(10))
else:
    print("No error records found for invoice analysis")

# C. Error concentration by country
print(f"\nC. ERROR DISTRIBUTION BY COUNTRY:")
if len(error_records) > 0:
    country_errors = error_records.groupby('Country_oms').agg({
        'Total_Value_variance': ['count', 'sum', 'mean'],
        'error_type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    }).round(2)
    
    country_errors.columns = ['Error_Count', 'Total_Financial_Impact', 'Avg_Error_Value', 'Most_Common_Error_Type']
    country_errors = country_errors.sort_values('Total_Financial_Impact', key=abs, ascending=False)
    print(country_errors.head())
else:
    print("No error records found for country analysis")

# 5. DETAILED ERROR BREAKDOWN BY TYPE
print(f"\n5. DETAILED ERROR BREAKDOWN BY TYPE\n")

for error_type in error_type_counts.index:
    if error_type != 'No Error':
        subset = df_matched[df_matched['error_type'] == error_type]
        financial_impact = subset['Total_Value_variance'].sum()
        avg_impact = subset['Total_Value_variance'].mean()
        
        print(f"{error_type}:")
        print(f"  Records: {len(subset):,}")
        print(f"  Total Financial Impact: £{financial_impact:,.2f}")
        print(f"  Average Impact per Record: £{avg_impact:.2f}")
        print(f"  Price Variance Range: £{subset['UnitPrice_variance'].min():.2f} to £{subset['UnitPrice_variance'].max():.2f}")
        
        # Show validation flags for this error type
        all_flags = []
        for flags in subset['validation_flags_oms'].tolist() + subset['validation_flags_wms'].tolist():
            if isinstance(flags, list):
                all_flags.extend(flags)
        
        if all_flags:
            flag_counts = pd.Series(all_flags).value_counts()
            print(f"  Associated Validation Flags: {dict(flag_counts)}")
        print()

# 6. SUMMARY STATISTICS
print(f"\n=== CLASSIFICATION SUMMARY ===\n")

total_errors = len(df_matched[df_matched['error_type'] != 'No Error'])
total_records = len(df_matched)
error_rate = (total_errors / total_records) * 100

print(f"Overall Error Statistics:")
print(f"  Total records processed: {total_records:,}")
print(f"  Records with errors: {total_errors:,}")
print(f"  Error rate: {error_rate:.2f}%")
print(f"  Total financial exposure: £{df_matched['Total_Value_variance'].sum():,.2f}")

print(f"\nError Type Breakdown:")
for error_type, count in error_type_counts.items():
    if error_type != 'No Error':
        percentage = (count / total_records) * 100
        financial_impact = df_matched[df_matched['error_type'] == error_type]['Total_Value_variance'].sum()
        print(f"  {error_type}: {count:,} records ({percentage:.2f}%) - £{financial_impact:,.2f}")

print(f"\nFinancial Risk Distribution:")
for risk_level in ['High Risk', 'Medium Risk', 'Low Risk']:
    if risk_level in df_matched['financial_impact'].values:
        count = (df_matched['financial_impact'] == risk_level).sum()
        percentage = (count / total_records) * 100
        financial_impact = df_matched[df_matched['financial_impact'] == risk_level]['Total_Value_variance'].sum()
        print(f"  {risk_level}: {count:,} records ({percentage:.2f}%) - £{financial_impact:,.2f}")

# 7. VALIDATION EFFECTIVENESS ANALYSIS
print(f"\n=== VALIDATION EFFECTIVENESS ANALYSIS ===\n")

# Check how many errors were caught by validation flags vs missed
errors_with_flags = 0
errors_without_flags = 0

for idx, row in df_matched[df_matched['error_type'] != 'No Error'].iterrows():
    flags_oms = row['validation_flags_oms'] if isinstance(row['validation_flags_oms'], list) else []
    flags_wms = row['validation_flags_wms'] if isinstance(row['validation_flags_wms'], list) else []
    
    if len(flags_oms) > 0 or len(flags_wms) > 0:
        errors_with_flags += 1
    else:
        errors_without_flags += 1

if total_errors > 0:
    detection_rate = (errors_with_flags / total_errors) * 100
    print(f"Validation Detection Effectiveness:")
    print(f"  Errors detected by validation flags: {errors_with_flags:,} ({detection_rate:.1f}%)")
    print(f"  Errors missed by validation: {errors_without_flags:,} ({100-detection_rate:.1f}%)")
    print(f"  Overall validation effectiveness: {detection_rate:.1f}%")
else:
    print("No errors found to analyze validation effectiveness")


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.
=== ERROR DETECTION & ROOT CAUSE ANALYSIS ===

STEP 1: AGGREGATING DUPLICATES FOR CONSISTENT ANALYSIS

OMS after aggregation: 536478 unique business transactions
WMS after aggregation: 536478 unique business transactions

STEP 2: CREATING MATCHED DATASET FOR ERROR ANALYSIS

Matched records for analysis: 536,478
Total financial variance: £-17,857.56

=== ERROR CATEGORIZATION & CLASSIFICATION ===

1. SYSTEMATIC GROUPING BY ERROR TYPE

Error Distribution by Type:
error_type
No Error                526634
System Sync Error         5350
Data Quality Error        3511
Business Logic Error       983
Name: count, dtype: int64

Total records analyzed: 536,478

2. FINANCIAL IMPACT SCORING

Financial Impact Classification:
                 Total_Value_variance                  UnitPrice_variance  \
         

**Observation:**

**Error Distribution & Impact Hierarchy:**
- The analysis successfully categorized 9,844 error records (1.83% error rate) across 536,478 unique business transactions, maintaining consistency with the Comparison phase results.
- Data Quality Errors emerge as the primary concern, representing only 0.65% of transactions but accounting for 99.5% of financial exposure (-£17,945.12 out of -£17,857.56 total variance).
- System Sync Errors show high volume (5,350 records, 1.00% of transactions) but minimal financial impact (£0.74), confirming that duplicate record aggregation effectively neutralizes their monetary effect while preserving audit visibility.

**Financial Risk Concentration:**
- High Risk transactions (477 records, 0.09%) concentrate 89% of total financial exposure (-£15,906.28), indicating that a small number of transactions drive the majority of reconciliation variance.
- The risk distribution follows a classic Pareto pattern: 0.18% of all transactions (High + Medium Risk) account for 99.7% of financial impact, enabling targeted remediation efforts.

**Geographic & Product Patterns:**
- United Kingdom dominates error volume (9,522 records, 96.7% of all errors) but shows relatively low average impact (-£1.51 per error), suggesting systematic but minor issues.
Netherlands and Australia exhibit high-severity, low-volume patterns (-£43.55 and -£64.31 average impact respectively), indicating concentrated data quality problems requiring immediate attention.
- Product concentration reveals that specific stock codes (22492, M, 85099B) consistently appear in high-impact errors, suggesting either product-specific data integration issues or systematic problems with particular inventory categories.

**Validation System Performance:**
- 90.1% detection effectiveness demonstrates robust validation coverage, with the system successfully flagging 8,867 out of 9,844 actual errors.
- The 9.9% gap (977 undetected errors) primarily consists of Business Logic Errors (small price variances) that fall below current validation thresholds, representing opportunities for enhanced detection algorithms.
- Missing_UnitPrice flags show perfect correlation with Data Quality Errors, confirming that the validation system accurately identifies the most financially significant issues.

**Business Impact Assessment:**
- The error concentration in specific invoices (C-prefixed invoices showing high positive variances) suggests potential credit note or return processing discrepancies that require process review.
- System Sync Errors correlating with Duplicate_Record flags (10,197 flags across 5,350 records) indicates successful duplicate detection and aggregation, preventing inflation of financial impact while maintaining audit trails.

**Strategic Implications:**
- Immediate Priority: Address 477 High Risk transactions (-£15,906.28 exposure) through targeted data quality improvements
- Medium-term Focus: Investigate Netherlands and Australia data integration processes
- Long-term Enhancement: Refine validation thresholds to capture the 9.9% of currently undetected errors, particularly small price variances that may indicate systematic rounding or currency conversion issues

This classification provides a clear roadmap for prioritized remediation efforts, focusing resources on the highest-impact issues while maintaining comprehensive visibility across all error categories.

# 2. Root Cause Analysis

In [37]:
print("=== ROOT CAUSE ANALYSIS ===\n")

# STEP 1: ROOT CAUSES BY ERROR TYPE
print("1. ROOT CAUSES BY ERROR TYPE\n")

error_records = df_matched[df_matched['error_type'] != 'No Error']

# Data Quality Errors
data_quality = df_matched[df_matched['error_type'] == 'Data Quality Error']
missing_prices = (data_quality['UnitPrice_variance'] <= -1.0).sum()
date_errors = data_quality['validation_flags_wms'].apply(
    lambda x: 'Invalid_Date' in x if isinstance(x, list) else False).sum()

print(f"Data Quality Errors ({len(data_quality):,} records):")
print(f"  - Missing prices: {missing_prices:,} records (ETL failure)")
print(f"  - Date corruption: {date_errors:,} records (migration error)")
print(f"  - Financial impact: £{data_quality['Total_Value_variance'].sum():,.2f}")

# System Sync Errors  
sync_errors = df_matched[df_matched['error_type'] == 'System Sync Error']
print(f"\nSystem Sync Errors ({len(sync_errors):,} records):")
print(f"  - Duplicate processing: {len(sync_errors):,} records")
print(f"  - Cause: Message queue reprocessing")
print(f"  - Financial impact: £{sync_errors['Total_Value_variance'].sum():,.2f}")

# Business Logic Errors
business_errors = df_matched[df_matched['error_type'] == 'Business Logic Error']
print(f"\nBusiness Logic Errors ({len(business_errors):,} records):")
print(f"  - Price rounding: +£0.01 adjustments")
print(f"  - Cause: Currency/tax calculation differences")
print(f"  - Financial impact: £{business_errors['Total_Value_variance'].sum():,.2f}")

# STEP 2: PATTERN ANALYSIS
print(f"\n\n2. KEY PATTERNS IDENTIFIED\n")

# Top affected products
print("Most Affected Products:")
product_errors = error_records.groupby('StockCode')['Total_Value_variance'].sum().sort_values(key=abs, ascending=False)
print(product_errors.head(5))

# Country distribution
print(f"\nCountry Error Distribution:")
country_errors = error_records.groupby('Country_oms').agg({
    'Total_Value_variance': ['count', 'sum']
}).round(2)
country_errors.columns = ['Error_Count', 'Total_Impact']
print(country_errors.sort_values('Total_Impact', key=abs, ascending=False).head(3))

# STEP 3: CORRELATION ANALYSIS
print(f"\n\n3. ERROR CORRELATIONS\n")

# Flag correlation with error types
print("Validation Flag Effectiveness:")
for error_type in ['Data Quality Error', 'System Sync Error', 'Business Logic Error']:
    subset = df_matched[df_matched['error_type'] == error_type]
    flagged = subset.apply(lambda row: 
        len(row['validation_flags_oms'] if isinstance(row['validation_flags_oms'], list) else []) > 0 or
        len(row['validation_flags_wms'] if isinstance(row['validation_flags_wms'], list) else []) > 0, axis=1).sum()
    
    detection_rate = (flagged / len(subset) * 100) if len(subset) > 0 else 0
    print(f"  {error_type}: {detection_rate:.1f}% detected by validation")

# STEP 4: SUMMARY & RECOMMENDATIONS
print(f"\n\n=== SUMMARY & RECOMMENDATIONS ===\n")

print("PRIMARY ROOT CAUSES:")
print("1. ETL Process Failure → Missing prices (£-17,945 impact)")
print("2. System Migration Error → Date corruption (11 records)")  
print("3. Message Queue Issues → Duplicate processing (5,350 records)")
print("4. Calculation Differences → Price rounding (983 records)")

print(f"\nIMMEDiate ACTIONS:")
print("1. Fix ETL job for price synchronization")
print("2. Investigate message queue retry logic") 
print("3. Review currency conversion algorithms")
print("4. Audit system migration scripts")

print(f"\nVALIDATION GAPS:")
total_errors = len(error_records)
undetected = 977  # from previous analysis
print(f"- {undetected:,} errors ({undetected/total_errors*100:.1f}%) remain undetected")
print("- Enhance validation for small price variances")


=== ROOT CAUSE ANALYSIS ===

1. ROOT CAUSES BY ERROR TYPE

Data Quality Errors (3,511 records):
  - Missing prices: 772 records (ETL failure)
  - Date corruption: 11 records (migration error)
  - Financial impact: £-17,945.12

System Sync Errors (5,350 records):
  - Duplicate processing: 5,350 records
  - Cause: Message queue reprocessing
  - Financial impact: £0.74

Business Logic Errors (983 records):
  - Price rounding: +£0.01 adjustments
  - Cause: Currency/tax calculation differences
  - Financial impact: £86.82


2. KEY PATTERNS IDENTIFIED

Most Affected Products:
StockCode
22492    -764.54
M         551.14
85099B   -484.76
22947    -465.64
21556    -404.98
Name: Total_Value_variance, dtype: float64

Country Error Distribution:
                Error_Count  Total_Impact
Country_oms                              
UNITED KINGDOM         9522     -14372.32
NETHERLANDS              21       -914.64
GERMANY                  61       -465.10


3. ERROR CORRELATIONS

Validation Flag Effec

Observations:
**Root Cause Hierarchy and Business Impact:**
- ETL Process Failure emerges as the dominant root cause, accounting for 772 missing price records and driving 99.2% of total financial exposure (-£17,945.12 out of -£17,857.56). This represents a critical system integration failure where price synchronization between OMS and WMS has broken down, likely due to API timeouts, database connectivity issues, or ETL job scheduling conflicts.
- System Migration Error shows surgical precision impact - only 11 records affected by date corruption, but the "ERR_DATE_2024" pattern suggests a systematic failure during system upgrade or data migration scripts. While financially negligible, this represents a compliance and audit trail risk that could escalate during regulatory reviews.
- Message Queue Reprocessing affects the highest volume (5,350 records) but demonstrates effective financial impact mitigation through duplicate aggregation. The near-zero financial impact (£0.74) confirms that the reconciliation methodology successfully neutralizes volume-based discrepancies while preserving audit visibility.

**Product and Geographic Risk Concentration:**
- Product-level analysis reveals concentrated risk patterns with StockCode 22492 showing the highest individual impact (-£764.54) and product "M" demonstrating positive variance (+£551.14), suggesting either product-specific pricing rules or category-based processing differences that require targeted investigation.
- Geographic distribution follows expected patterns with UK representing 96.7% of error volume (9,522 records) but relatively low average impact (-£1.51 per error). However, Netherlands shows disproportionate severity (-£43.55 average impact) despite low volume (21 records), indicating potential currency conversion or international pricing synchronization issues requiring immediate attention.

**Validation System Performance Assessment:**
- Perfect detection rates for Data Quality and System Sync Errors (100% each) demonstrate robust validation coverage for high-impact issues, confirming that the validation framework successfully identifies the most financially significant problems before they reach production reconciliation.
- Business Logic Error detection gap (0.6% detection rate) represents the primary validation enhancement opportunity. The 983 undetected price rounding errors, while individually small (+£0.01), collectively represent systematic calculation differences that may indicate underlying currency conversion, tax calculation, or pricing algorithm discrepancies requiring algorithmic validation rules rather than data quality checks.

**Strategic Implications and Remediation Priority:**
- Immediate Priority (ETL Failure): The concentration of 99.2% financial impact in ETL process failures demands urgent intervention. Investigation should focus on ETL job monitoring, API endpoint health checks, and database connection pooling to prevent future price synchronization failures.
- Medium-term Focus (International Operations): Netherlands' disproportionate error severity suggests international pricing and currency conversion processes require systematic review, potentially indicating broader issues with multi-currency operations that could affect other international markets.

The 9.9% validation gap, primarily in Business Logic Errors, indicates opportunity for enhanced algorithmic validation. Implementing tolerance-based price variance detection and automated currency conversion validation could capture the remaining undetected errors while maintaining operational efficiency.

The 90.1% overall validation effectiveness, combined with clear root cause identification and quantified business impact, demonstrates a mature reconciliation process capable of supporting data-driven remediation decisions and continuous improvement initiatives.

**About noise created in the initial part of project:**
- 500 intentional duplicates → 5,350 System Sync Errors detected
- 1,000 missing prices → 772 Data Quality Errors detected (post-aggregation)
- 11 date corruptions → 11 Invalid_Date flags detected
- 1,000 price modifications → 983 Business Logic Errors detected

